# PPO 最小代码实验

这个 notebook 仿照 `dpo.ipynb`，用一个很小的 MiniMind 模型拆解 PPO 在语言模型训练中的核心数据流。

为了便于学习，这里不会调用完整的在线生成和真实 Reward Model，而是用一条固定回答模拟 rollout 结果，并用几条显式规则临时代替 Reward Model。

每个代码格只推进一小步，并打印刚得到的变量，方便按顺序观察 PPO 的张量流。

> 注意：真实 PPO / RLAIF 训练仍然需要 Reward Model 或其它奖励来源。这个 notebook 省略真实 Reward Model，只是为了先把 PPO 的张量流和 loss 计算讲清楚。


## 1. 构造 PPO / RLAIF 最小训练样本

PPO 和 DPO 的输入形式不同：

- DPO 输入是 `chosen/rejected` 偏好对；
- PPO 输入通常是一个 prompt，模型先根据 prompt 生成 response，再通过 reward 函数或 Reward Model 打分。


In [ ]:
from pathlib import Path
import json


ppo_train_sample = {
    "conversations": [
        {
            "role": "user",
            "content": "请用两句话解释 PPO 为什么要使用裁剪目标。",
        },
        {
            "role": "assistant",
            "content": "PPO 使用裁剪目标是为了限制新策略相对旧策略变化过大，从而让训练更稳定。这样即使某次 reward 较高，模型也不会因为单个样本被过度更新。",
        },
    ]
}

print("样本条数:", 1)
print("user:", ppo_train_sample["conversations"][0]["content"])
print("assistant:", ppo_train_sample["conversations"][1]["content"])


In [ ]:
current_dir = Path.cwd()

if (current_dir / "model" / "tokenizer.json").exists():
    project_root = current_dir
elif (current_dir.parent / "model" / "tokenizer.json").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("找不到 model/tokenizer.json，请在 minimind 项目根目录或 notebook 目录运行。")

print("当前目录:", current_dir)
print("项目根目录:", project_root)


In [ ]:
output_path = project_root / "dataset" / "ppo_toy.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    f.write(json.dumps(ppo_train_sample, ensure_ascii=False) + "\n")

print("已生成:", output_path)
print(json.dumps(ppo_train_sample, ensure_ascii=False, indent=2))


## 2. 使用 tokenizer 构造 prompt 和 rollout 序列

真实 PPO 训练会先让 actor 根据 prompt 采样生成 response。为了让这个 notebook 更稳定，这里直接把样本中的 assistant 回复当作一次已经生成好的 rollout。


In [ ]:
import torch
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(str(project_root / "model"))
max_length = 160

pad_token_id = tokenizer.pad_token_id
if pad_token_id is None:
    pad_token_id = tokenizer.eos_token_id

print("vocab size:", tokenizer.vocab_size)
print("eos token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("pad token id:", pad_token_id)
print("max_length:", max_length)


In [ ]:
prompt = tokenizer.apply_chat_template(
    ppo_train_sample["conversations"][:-1],
    tokenize=False,
    add_generation_prompt=True,
    open_thinking=False,
)

print("prompt 只包含用户问题和 assistant 生成起点:")
print(prompt)


In [ ]:
response_text = ppo_train_sample["conversations"][-1]["content"]

print("固定 response 用来模拟 actor rollout:")
print(response_text)


In [ ]:
prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
response_ids = tokenizer(response_text + tokenizer.eos_token + "\n", add_special_tokens=False).input_ids

print("prompt token 数:", len(prompt_ids))
print("response token 数:", len(response_ids))
print("prompt 前 10 个 token id:", prompt_ids[:10])
print("response 前 10 个 token id:", response_ids[:10])


In [ ]:
input_ids = (prompt_ids + response_ids)[:max_length]
gen_out = torch.tensor([input_ids], dtype=torch.long)
full_mask = torch.ones_like(gen_out, dtype=torch.long)
prompt_length = len(prompt_ids)

print("gen_out 模拟 rollout_engine.rollout(...) 的 output_ids")
print("gen_out shape:", tuple(gen_out.shape))
print("full_mask shape:", tuple(full_mask.shape))
print("prompt_length:", prompt_length)
print("完整序列预览:")
print(tokenizer.decode(gen_out[0].tolist()[:80]))


## 3. 创建 actor、ref 和 critic 模型

PPO 比 DPO 多了一个 critic：

- `actor_model`：当前要训练的策略模型；
- `ref_model`：冻结参考模型，用来限制 actor 不要偏离太远；
- `critic_backbone + critic_value_head`：价值模型，估计每个 token 位置后续能拿到多少 reward。


In [ ]:
from copy import deepcopy
import sys

from torch import nn
import torch.nn.functional as F


if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

torch.manual_seed(42)

print("device:", device)
print("已导入 MiniMindConfig 和 MiniMindForCausalLM")


In [ ]:
lm_config = MiniMindConfig(
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=max_length,
    flash_attn=False,
    dropout=0.0,
    use_moe=False,
)

print("tiny MiniMind 配置:")
print("hidden_size:", lm_config.hidden_size)
print("num_hidden_layers:", lm_config.num_hidden_layers)
print("num_attention_heads:", lm_config.num_attention_heads)
print("vocab_size:", lm_config.vocab_size)


In [ ]:
actor_model = MiniMindForCausalLM(lm_config).to(device)
ref_model = deepcopy(actor_model).to(device)

critic_backbone = MiniMindForCausalLM(lm_config).to(device)
critic_value_head = nn.Linear(lm_config.hidden_size, 1).to(device)
critic_params = list(critic_backbone.parameters()) + list(critic_value_head.parameters())

actor_param_count = sum(p.numel() for p in actor_model.parameters())
critic_param_count = sum(p.numel() for p in critic_params)

print("actor 参数量:", f"{actor_param_count / 1e6:.3f}M")
print("critic 参数量:", f"{critic_param_count / 1e6:.3f}M")
print("critic = MiniMind backbone + value head")


In [ ]:
actor_model.train()
critic_backbone.train()
critic_value_head.train()
ref_model.eval()

for param in ref_model.parameters():
    param.requires_grad_(False)

actor_optimizer = torch.optim.AdamW(actor_model.parameters(), lr=1e-4)
critic_optimizer = torch.optim.AdamW(critic_params, lr=2e-4)

gen_out = gen_out.to(device)
full_mask = full_mask.to(device)

print("ref_model 已冻结:", not any(p.requires_grad for p in ref_model.parameters()))
print("actor_optimizer lr:", actor_optimizer.param_groups[0]["lr"])
print("critic_optimizer lr:", critic_optimizer.param_groups[0]["lr"])
print("gen_out device:", gen_out.device)


## 4. 标出 response 区域

PPO 只优化模型生成的 response token，不优化用户 prompt。这里先准备 next-token labels，再做 response mask。


In [ ]:
labels = gen_out[:, 1:].clone()
seq_len = gen_out.size(1) - 1
resp_start = prompt_length - 1

print("gen_out: [t0, t1, t2, ...]")
print("labels:  [t1, t2, t3, ...]")
print("labels shape:", tuple(labels.shape))
print("seq_len:", seq_len)
print("response 的第一个 label 位置 resp_start:", resp_start)


In [ ]:
token_positions = torch.arange(seq_len, device=device).unsqueeze(0)
resp_mask = token_positions >= resp_start
non_pad_mask = ~labels.eq(pad_token_id)
final_mask = (resp_mask & non_pad_mask).float()

resp_policy_mask = final_mask[:, resp_start:]
resp_value_mask = resp_policy_mask.clone()

print("final_mask shape:", tuple(final_mask.shape))
print("response token 数:", int(resp_policy_mask.sum().item()))
print("resp_policy_mask shape:", tuple(resp_policy_mask.shape))
print("mask 前 20 位:", final_mask[0, :20].int().tolist())


## 5. 记录 old log prob 和 old value

PPO 是 on-policy 算法。先用旧策略采样一批 response，然后保存当时的 `old_logp` 和 `old_values`。后续更新 actor 时，会比较新旧策略的概率比值 `ratio = exp(new_logp - old_logp)`。


In [ ]:
with torch.no_grad():
    old_actor_outputs = actor_model(input_ids=gen_out, attention_mask=full_mask)
    old_actor_logits = old_actor_outputs.logits[:, :-1]
    old_actor_log_probs = F.log_softmax(old_actor_logits, dim=-1)
    old_logp_all = old_actor_log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    old_resp_logp = old_logp_all[:, resp_start:] * resp_policy_mask

print("old_logp_all shape:", tuple(old_logp_all.shape))
print("old_resp_logp shape:", tuple(old_resp_logp.shape))
print("old_resp_logp 前 5 个:", old_resp_logp[0, :5].detach().cpu().tolist())


In [ ]:
with torch.no_grad():
    ref_outputs = ref_model(input_ids=gen_out, attention_mask=full_mask)
    ref_logits = ref_outputs.logits[:, :-1]
    ref_log_probs = F.log_softmax(ref_logits, dim=-1)
    ref_logp_all = ref_log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    ref_resp_logp = ref_logp_all[:, resp_start:] * resp_policy_mask

print("ref_resp_logp shape:", tuple(ref_resp_logp.shape))
print("ref 和 old actor 初始来自同一份权重")
print("平均差值:", float(((ref_resp_logp - old_resp_logp) * resp_policy_mask).mean().cpu()))


In [ ]:
with torch.no_grad():
    critic_hidden, _, _ = critic_backbone.model(input_ids=gen_out, attention_mask=full_mask)
    values_seq = critic_value_head(critic_hidden).squeeze(-1)
    old_resp_values = values_seq[:, resp_start:-1] * resp_value_mask

print("values_seq shape:", tuple(values_seq.shape))
print("old_resp_values shape:", tuple(old_resp_values.shape))
print("old_resp_values 前 5 个:", old_resp_values[0, :5].detach().cpu().tolist())


## 6. 准备 reward、advantage 和 return

真实 `trainer/train_ppo.py` 会调用 Reward Model 给回答打分；这里为了不依赖大模型权重，把 reward 的每一项规则直接写出来。


In [ ]:
length_reward = 0.5 if 20 <= len(response_text) <= 200 else -0.5
clip_keyword_reward = 0.5 if "裁剪" in response_text else -0.2
stable_keyword_reward = 0.5 if "稳定" in response_text else -0.2

reward_value = length_reward + clip_keyword_reward + stable_keyword_reward
reward = torch.tensor([reward_value], device=device)

print("长度奖励:", length_reward)
print("包含'裁剪'奖励:", clip_keyword_reward)
print("包含'稳定'奖励:", stable_keyword_reward)
print("总 reward:", float(reward.cpu()))


In [ ]:
token_rewards = torch.zeros_like(old_resp_logp)
response_lengths = resp_policy_mask.sum(dim=1).long().clamp(min=1)
last_idx = response_lengths - 1
batch_index = torch.arange(gen_out.size(0), device=device)
token_rewards[batch_index, last_idx] = reward

print("token_rewards shape:", tuple(token_rewards.shape))
print("response_lengths:", response_lengths.detach().cpu().tolist())
print("reward 放在最后一个 response token index:", last_idx.detach().cpu().tolist())
print("非零 token reward:", token_rewards[token_rewards != 0].detach().cpu().tolist())


In [ ]:
gamma = 1.0
lam = 0.95

gen_len = old_resp_values.size(1)
lastgaelam = torch.zeros(gen_out.size(0), device=device)
advs_rev = []

for t in reversed(range(gen_len)):
    next_value = old_resp_values[:, t + 1] if t < gen_len - 1 else 0.0
    delta = token_rewards[:, t] + gamma * next_value - old_resp_values[:, t]
    lastgaelam = delta + gamma * lam * lastgaelam
    advs_rev.append(lastgaelam)

advantages = torch.stack(advs_rev[::-1], dim=1)

print("GAE gamma:", gamma)
print("GAE lambda:", lam)
print("advantages shape:", tuple(advantages.shape))
print("advantages 前 5 个:", advantages[0, :5].detach().cpu().tolist())


In [ ]:
returns = advantages + old_resp_values

adv_mean = (advantages * resp_policy_mask).sum() / resp_policy_mask.sum().clamp(min=1)
adv_var = ((advantages - adv_mean) ** 2 * resp_policy_mask).sum() / resp_policy_mask.sum().clamp(min=1)
advantages = (advantages - adv_mean) * torch.rsqrt(adv_var + 1e-8) * resp_policy_mask

print("returns shape:", tuple(returns.shape))
print("标准化前 advantage mean:", float(adv_mean.detach().cpu()))
print("标准化前 advantage var:", float(adv_var.detach().cpu()))
print("标准化后 advantage mean:", float(((advantages * resp_policy_mask).sum() / resp_policy_mask.sum()).detach().cpu()))


## 7. 计算 PPO clipped loss

PPO 的核心是限制新策略不要离旧策略太远。它不是直接最大化 `advantage * new_logp`，而是比较新旧策略概率比 `ratio`，并把 ratio 裁剪到 `[1 - clip_epsilon, 1 + clip_epsilon]`。


In [ ]:
clip_epsilon = 0.2
cliprange_value = 0.2
vf_coef = 0.5
kl_coef = 0.02

print("clip_epsilon:", clip_epsilon)
print("cliprange_value:", cliprange_value)
print("vf_coef:", vf_coef)
print("kl_coef:", kl_coef)


In [ ]:
actor_outputs = actor_model(input_ids=gen_out, attention_mask=full_mask)
new_logits = actor_outputs.logits[:, :-1]
new_log_probs = F.log_softmax(new_logits, dim=-1)
new_logp_all = new_log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
new_resp_logp = new_logp_all[:, resp_start:]

print("new_logp_all shape:", tuple(new_logp_all.shape))
print("new_resp_logp shape:", tuple(new_resp_logp.shape))
print("当前还没更新，所以 new 和 old 很接近")
print("平均 new-old:", float(((new_resp_logp - old_resp_logp) * resp_policy_mask).mean().detach().cpu()))


In [ ]:
critic_hidden_new, _, _ = critic_backbone.model(input_ids=gen_out, attention_mask=full_mask)
new_values_seq = critic_value_head(critic_hidden_new).squeeze(-1)
new_resp_values = new_values_seq[:, resp_start:-1]

print("new_values_seq shape:", tuple(new_values_seq.shape))
print("new_resp_values shape:", tuple(new_resp_values.shape))
print("new_resp_values 前 5 个:", new_resp_values[0, :5].detach().cpu().tolist())


In [ ]:
log_ratio = (new_resp_logp - old_resp_logp) * resp_policy_mask
ratio = torch.exp(log_ratio)

print("ratio shape:", tuple(ratio.shape))
print("ratio min:", float((ratio * resp_policy_mask).masked_select(resp_policy_mask.bool()).min().detach().cpu()))
print("ratio max:", float((ratio * resp_policy_mask).masked_select(resp_policy_mask.bool()).max().detach().cpu()))
print("ratio 前 5 个:", ratio[0, :5].detach().cpu().tolist())


In [ ]:
unclipped_policy_loss = -advantages * ratio
clipped_ratio = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon)
clipped_policy_loss = -advantages * clipped_ratio

policy_loss_before_kl = (
    torch.max(unclipped_policy_loss, clipped_policy_loss) * resp_policy_mask
).sum() / resp_policy_mask.sum().clamp(min=1)

print("unclipped_policy_loss shape:", tuple(unclipped_policy_loss.shape))
print("clipped_ratio 范围:", float(clipped_ratio.min().detach().cpu()), float(clipped_ratio.max().detach().cpu()))
print("policy_loss_before_kl:", float(policy_loss_before_kl.detach().cpu()))


In [ ]:
ref_minus_new = ref_resp_logp - new_resp_logp
kl_ref_penalty = (
    (torch.exp(ref_minus_new) - ref_minus_new - 1.0) * resp_policy_mask
).sum() / resp_policy_mask.sum().clamp(min=1)

policy_loss = policy_loss_before_kl + kl_coef * kl_ref_penalty

print("kl_ref_penalty:", float(kl_ref_penalty.detach().cpu()))
print("policy_loss = policy_loss_before_kl + kl_coef * kl_ref_penalty")
print("policy_loss:", float(policy_loss.detach().cpu()))


In [ ]:
value_loss_unclipped = (new_resp_values - returns) ** 2
value_delta = torch.clamp(new_resp_values - old_resp_values, -cliprange_value, cliprange_value)
value_clipped = old_resp_values + value_delta
value_loss_clipped = (value_clipped - returns) ** 2

value_loss = 0.5 * (
    torch.max(value_loss_unclipped, value_loss_clipped) * resp_value_mask
).sum() / resp_value_mask.sum().clamp(min=1)

print("value_loss_unclipped mean:", float((value_loss_unclipped * resp_value_mask).sum().detach().cpu() / resp_value_mask.sum().detach().cpu()))
print("value_loss_clipped mean:", float((value_loss_clipped * resp_value_mask).sum().detach().cpu() / resp_value_mask.sum().detach().cpu()))
print("value_loss:", float(value_loss.detach().cpu()))


In [ ]:
approx_kl = (
    0.5 * (log_ratio ** 2) * resp_policy_mask
).sum() / resp_policy_mask.sum().clamp(min=1)

clipfrac = (
    (((ratio - 1.0).abs() > clip_epsilon).float() * resp_policy_mask).sum()
    / resp_policy_mask.sum().clamp(min=1)
)

print("approx_kl:", float(approx_kl.detach().cpu()))
print("clipfrac:", float(clipfrac.detach().cpu()))


In [ ]:
aux_loss = actor_outputs.aux_loss if lm_config.use_moe else torch.tensor(0.0, device=device)
loss = policy_loss + vf_coef * value_loss + aux_loss

print("policy_loss:", float(policy_loss.detach().cpu()))
print("vf_coef * value_loss:", float((vf_coef * value_loss).detach().cpu()))
print("aux_loss:", float(aux_loss.detach().cpu()))
print("total loss:", float(loss.detach().cpu()))


## 8. 可选：执行一次反向传播

下面几个格子展示 PPO loss 如何真正更新 actor 和 critic。教学时可以先只运行到上一节，理解 loss；确认无误后再继续观察参数更新闭环。


In [ ]:
actor_optimizer.zero_grad(set_to_none=True)
critic_optimizer.zero_grad(set_to_none=True)

print("已清空 actor 和 critic 的历史梯度")


In [ ]:
loss.backward()

actor_grad_norm = torch.nn.utils.clip_grad_norm_(actor_model.parameters(), max_norm=1.0)
critic_grad_norm = torch.nn.utils.clip_grad_norm_(critic_params, max_norm=1.0)

print("已完成 loss.backward()")
print("actor grad norm before clip:", float(actor_grad_norm.detach().cpu()))
print("critic grad norm before clip:", float(critic_grad_norm.detach().cpu()))


In [ ]:
actor_optimizer.step()
critic_optimizer.step()

print("完成一次 PPO 风格的 actor / critic 更新")
